In [1]:
import hashlib
import json
import os

In [2]:
folder_path = "/content/integrity_test"

os.makedirs(folder_path, exist_ok=True)

print("Test folder created:", folder_path)

Test folder created: /content/integrity_test


In [3]:
test_files = {
    "document1.txt": "This is the original content of document 1.",
    "document2.txt": "This is the original content of document 2.",
    "notes.txt": "Cyber Security Internship - InfozaTech"
}

for filename, content in test_files.items():

    file_path = os.path.join(folder_path, filename)

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(content)

print("Test files created successfully.")

Test files created successfully.


In [4]:
print("Files currently being monitored:")

for filename in os.listdir(folder_path):
    print("-", filename)

Files currently being monitored:
- document2.txt
- notes.txt
- document1.txt


In [5]:
def calculate_file_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(4096)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [6]:
test_file = os.path.join(folder_path, "document1.txt")

hash_value = calculate_file_hash(test_file)

print("SHA-256:")
print(hash_value)

SHA-256:
0858a1ed240fdb04eb267bb08834360fd841298a6c92b5e5b92dc0f85b743a7f


In [7]:
def scan_folder(folder_path):

    hashes = {}

    for root, directories, files in os.walk(folder_path):

        for filename in files:

            file_path = os.path.join(root, filename)

            relative_path = os.path.relpath(
                file_path,
                folder_path
            )

            hashes[relative_path] = calculate_file_hash(file_path)

    return hashes

In [8]:
current_hashes = scan_folder(folder_path)

for filename, file_hash in current_hashes.items():

    print(filename)
    print(file_hash)
    print()

document2.txt
030c0ea92a053a253d8b8095a537a7ada2a9961764c70c6c691684dfa2c0038d

notes.txt
06d323fe22f7c800702d3b8394807c3f82afbf2304353642ad8c0b5b2e1e98b0

document1.txt
0858a1ed240fdb04eb267bb08834360fd841298a6c92b5e5b92dc0f85b743a7f



In [9]:
baseline_file = "/content/baseline.json"

In [10]:
baseline_hashes = scan_folder(folder_path)

with open(baseline_file, "w", encoding="utf-8") as file:
    json.dump(baseline_hashes, file, indent=4)

print("Baseline snapshot created successfully.")

Baseline snapshot created successfully.


In [11]:
print("BASELINE SNAPSHOT")
print("=" * 60)

for filename, file_hash in baseline_hashes.items():

    print(f"\nFile: {filename}")
    print(f"SHA-256: {file_hash}")

BASELINE SNAPSHOT

File: document2.txt
SHA-256: 030c0ea92a053a253d8b8095a537a7ada2a9961764c70c6c691684dfa2c0038d

File: notes.txt
SHA-256: 06d323fe22f7c800702d3b8394807c3f82afbf2304353642ad8c0b5b2e1e98b0

File: document1.txt
SHA-256: 0858a1ed240fdb04eb267bb08834360fd841298a6c92b5e5b92dc0f85b743a7f


## 6. Simulate File Changes

To test the integrity checker, three controlled changes are made:

1. `document1.txt` is modified.
2. `document2.txt` is deleted.
3. `new_file.txt` is added.

The next scan should identify all three changes correctly.

In [12]:
# MODIFY document1.txt

modified_file = os.path.join(
    folder_path,
    "document1.txt"
)

with open(modified_file, "w", encoding="utf-8") as file:
    file.write(
        "This file has been MODIFIED after the baseline was created."
    )


# DELETE document2.txt

deleted_file = os.path.join(
    folder_path,
    "document2.txt"
)

if os.path.exists(deleted_file):
    os.remove(deleted_file)


# ADD a new file

new_file = os.path.join(
    folder_path,
    "new_file.txt"
)

with open(new_file, "w", encoding="utf-8") as file:
    file.write(
        "This file was ADDED after the baseline was created."
    )


print("Test changes completed.")

Test changes completed.


In [13]:
current_hashes = scan_folder(folder_path)

print("Folder re-scanned successfully.")

Folder re-scanned successfully.


In [14]:
def compare_hashes(baseline, current):

    added = []
    deleted = []
    modified = []
    unchanged = []

    # Check current files
    for filename in current:

        if filename not in baseline:

            added.append(filename)

        elif current[filename] != baseline[filename]:

            modified.append(filename)

        else:

            unchanged.append(filename)

    # Check for deleted files
    for filename in baseline:

        if filename not in current:

            deleted.append(filename)

    return added, deleted, modified, unchanged

In [15]:
added, deleted, modified, unchanged = compare_hashes(
    baseline_hashes,
    current_hashes
)

In [16]:
print("=" * 55)
print("             FILE INTEGRITY REPORT")
print("=" * 55)

print("\n[+] Added Files:")
if added:
    for filename in added:
        print("   +", filename)
else:
    print("   None")


print("\n[-] Deleted Files:")
if deleted:
    for filename in deleted:
        print("   -", filename)
else:
    print("   None")


print("\n[!] Modified Files:")
if modified:
    for filename in modified:
        print("   !", filename)
else:
    print("   None")


print("\n[✓] Unchanged Files:")
if unchanged:
    for filename in unchanged:
        print("   ✓", filename)
else:
    print("   None")


print("\n" + "=" * 55)

             FILE INTEGRITY REPORT

[+] Added Files:
   + new_file.txt

[-] Deleted Files:
   - document2.txt

[!] Modified Files:
   ! document1.txt

[✓] Unchanged Files:
   ✓ notes.txt

